In [15]:
# --- Path plumbing (point to src) + autoreload ---
from pathlib import Path
import sys
import pandas as pd

%load_ext autoreload
%autoreload 2

NB   = Path.cwd()
ROOT = NB.parent
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from data_pipeline.config import ROOT as ROOT_CFG, PROCESSED_DIR, CLEANED
from data_pipeline.loaders import load_vol_surface_wrds, load_forward_prices_wrds, load_option_volume_wrds
print("CWD:", NB)
print("SRC:", SRC)
print("ROOT:", ROOT_CFG)
print("PROC:", PROCESSED_DIR)
print("CLEANED:", CLEANED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
CWD: /Users/ya/Desktop/deep-hedging-rl/notebooks
SRC: /Users/ya/Desktop/deep-hedging-rl/src
ROOT: /Users/ya/Desktop/deep-hedging-rl
PROC: /Users/ya/Desktop/deep-hedging-rl/data/processed
CLEANED: /Users/ya/Desktop/deep-hedging-rl/data/processed/cleaned


## Importing and checking the data

In [16]:
fp_market_spx = CLEANED / "market_plus_panel_spx.parquet"
market_spx = pd.read_parquet(fp_market_spx)
display(market_spx.head())

,close_gspc,vix,rate_10y,hvol_10d,hvol_14d,hvol_30d,hvol_60d,hvol_91d,hvol_122d,hvol_152d,...,hvol_365d,hvol_547d,hvol_730d,hvol_1825d,fwd_front,index,high_gspc,low_gspc,open_gspc,rv_21d
date,,,,,,,,,,,,,,,,,,,,,
1996-01-04,617.700012,13.78,5.65,0.066934,0.06226,0.099637,0.091679,0.088536,0.082083,0.07694,...,0.078532,0.08396,0.089418,0.102954,619.154402,11573,624.48999,613.960022,621.320007,0.125956
1996-01-04,617.700012,13.78,5.65,0.066934,0.06226,0.099637,0.091679,0.088536,0.082083,0.07694,...,0.078532,0.08396,0.089418,0.102954,620.656417,11573,624.48999,613.960022,621.320007,0.125956
1996-01-04,617.700012,13.78,5.65,0.066934,0.06226,0.099637,0.091679,0.088536,0.082083,0.07694,...,0.078532,0.08396,0.089418,0.102954,621.805572,11573,624.48999,613.960022,621.320007,0.125956
1996-01-04,617.700012,13.78,5.65,0.066934,0.06226,0.099637,0.091679,0.088536,0.082083,0.07694,...,0.078532,0.08396,0.089418,0.102954,626.574252,11573,624.48999,613.960022,621.320007,0.125956
1996-01-04,617.700012,13.78,5.65,0.066934,0.06226,0.099637,0.091679,0.088536,0.082083,0.07694,...,0.078532,0.08396,0.089418,0.102954,630.078895,11573,624.48999,613.960022,621.320007,0.125956


In [17]:
print(market_spx.isna().sum())

close_gspc    0
vix           0
rate_10y      0
hvol_10d      0
hvol_14d      0
hvol_30d      0
hvol_60d      0
hvol_91d      0
hvol_122d     0
hvol_152d     0
hvol_182d     0
hvol_273d     0
hvol_365d     0
hvol_547d     0
hvol_730d     0
hvol_1825d    0
fwd_front     0
index         0
high_gspc     0
low_gspc      0
open_gspc     0
rv_21d        0
dtype: int64


In [18]:
fp_market_spy = CLEANED / "market_plus_panel_spy.parquet"
market_spy = pd.read_parquet(fp_market_spy)
display(market_spy.head())

,close_spy,vix,rate_10y,hvol_10d,hvol_14d,hvol_30d,hvol_60d,hvol_91d,hvol_122d,hvol_152d,...,hvol_273d,hvol_365d,hvol_547d,hvol_730d,hvol_1825d,fwd_front,high_spy,low_spy,open_spy,rv_21d
date,,,,,,,,,,,,,,,,,,,,,
2005-01-21,79.452278,14.36,4.16,0.125623,0.118257,0.097512,0.089121,0.09803,0.10453,0.099892,...,0.108384,0.112556,0.112634,0.137524,0.203975,116.787588,80.282316,79.363833,80.139442,0.094577
2005-01-21,79.452278,14.36,4.16,0.125623,0.118257,0.097512,0.089121,0.09803,0.10453,0.099892,...,0.108384,0.112556,0.112634,0.137524,0.203975,117.001989,80.282316,79.363833,80.139442,0.094577
2005-01-21,79.452278,14.36,4.16,0.125623,0.118257,0.097512,0.089121,0.09803,0.10453,0.099892,...,0.108384,0.112556,0.112634,0.137524,0.203975,116.706210,80.282316,79.363833,80.139442,0.094577
2005-01-21,79.452278,14.36,4.16,0.125623,0.118257,0.097512,0.089121,0.09803,0.10453,0.099892,...,0.108384,0.112556,0.112634,0.137524,0.203975,117.040972,80.282316,79.363833,80.139442,0.094577
2005-01-21,79.452278,14.36,4.16,0.125623,0.118257,0.097512,0.089121,0.09803,0.10453,0.099892,...,0.108384,0.112556,0.112634,0.137524,0.203975,117.458144,80.282316,79.363833,80.139442,0.094577


In [19]:
market_spy = market_spy.reset_index()

In [20]:
market_spy.columns

Index(['date', 'close_spy', 'vix', 'rate_10y', 'hvol_10d', 'hvol_14d',
       'hvol_30d', 'hvol_60d', 'hvol_91d', 'hvol_122d', 'hvol_152d',
       'hvol_182d', 'hvol_273d', 'hvol_365d', 'hvol_547d', 'hvol_730d',
       'hvol_1825d', 'fwd_front', 'high_spy', 'low_spy', 'open_spy', 'rv_21d'],
      dtype='object')

In [21]:
print(market_spy.isna().sum())

date          0
close_spy     0
vix           0
rate_10y      0
hvol_10d      0
hvol_14d      0
hvol_30d      0
hvol_60d      0
hvol_91d      0
hvol_122d     0
hvol_152d     0
hvol_182d     0
hvol_273d     0
hvol_365d     0
hvol_547d     0
hvol_730d     0
hvol_1825d    0
fwd_front     0
high_spy      0
low_spy       0
open_spy      0
rv_21d        0
dtype: int64


In [22]:
import pyarrow.dataset as ds, pandas as pd, numpy as np
from pathlib import Path
from data_pipeline.config import ROOT

D = ds.dataset(ROOT / "data/processed/cleaned/SPX", format="parquet")
df = D.to_table(columns=["date","tenor_d","put_call","delta"]).to_pandas()

print("put_call uniques:", sorted(pd.Series(df["put_call"]).dropna().unique().tolist()))
print(df["delta"].describe())   # check if around ±1 or ±100

# candidates near 30D/25Δ with current tolerances
x = df.copy()
x = x[x["tenor_d"].sub(30).abs() <= 15]
x["abs_delta"] = x["delta"].abs()
cand = x[x["abs_delta"].sub(0.25).abs() <= 0.05]
print("rows near 30D & 25Δ:", len(cand), "| dates covered:", cand["date"].nunique())


put_call uniques: ['C', 'P']
count    2.950670e+07
mean     1.373999e-01
std      5.965577e-01
min     -9.999990e-01
25%     -1.969120e-01
50%     -1.986000e-03
75%      7.594440e-01
max      1.000000e+00
Name: delta, dtype: float64
rows near 30D & 25Δ: 192720 | dates covered: 5291


## Simulator

In [23]:
# --- 1) Build the simulation panel from your library (single source of truth) ---

from pathlib import Path
import pandas as pd
import numpy as np

from simulator.build_panel import build_sim_panel  # your high-level orchestrator

# Paths
ROOT = Path(ROOT)  # from data_pipeline.config import ROOT (already in your cell)
SPX_CLEAN = ROOT / "data" / "processed" / "cleaned" / "SPX"
#MP_SPY    = ROOT / "data" / "processed" / "cleaned" / "market_plus_panel_spy.parquet"

# Load market panel (SPY)
market_spy = market_spy.copy()

# Choose decision/realization convention:
ACT_AT_OPEN = False   # False → decide at close_t, realize close→close; True → open→open

# Guarded forward-fill limit for IV features (in business days)
FFILL_LIMIT_DAYS = 1

panel, state_cols = build_sim_panel(
    market_df=market_spy,
    spx_clean_dir=SPX_CLEAN,
    include_spy=False,            # keep False unless you also built SPY option features
    act_at_open=ACT_AT_OPEN,
    ffill_limit=FFILL_LIMIT_DAYS,
)

print("Built panel:", panel.shape, "| first:", panel['date'].min(), "| last:", panel['date'].max())
print("State cols:", state_cols)
panel.to_csv(CLEANED / "hedging_panel_spx.csv", index=False)


Built panel: (18018, 29) | first: 2005-01-21 00:00:00 | last: 2023-08-31 00:00:00
State cols: ['iv_atm_30d_spx', 'iv_ts_slope_spx', 'iv_skew_30d_spx', 'vix', 'rate_10y', 'rv_21d', 'hvol_30d', 'hvol_91d']


In [24]:
# --- 2) Train/valid/test split + scaler (fit on TRAIN only; no leakage) ---

TRAIN_END = pd.Timestamp("2017-12-31")
VALID_END = pd.Timestamp("2019-12-31")

mask_train = panel["date"] <= TRAIN_END
mask_valid = (panel["date"] > TRAIN_END) & (panel["date"] <= VALID_END)
mask_test  = panel["date"] > VALID_END

mu = panel.loc[mask_train, state_cols].mean()
sigma = panel.loc[mask_train, state_cols].std(ddof=1)
sigma = sigma.replace(0, np.nan).fillna(1.0)  # guard constants / all-NaN

# Window-wise scaler used by the env
scaler = lambda obs: (obs - mu.values) / sigma.values

print("Rows (train/valid/test):", mask_train.sum(), mask_valid.sum(), mask_test.sum())

Rows (train/valid/test): 7854 2505 7659


In [25]:
# --- 3) Env construction and baseline policy rollout ---

from simulator.env import HedgingEnv
from simulator.rewards import pnl_only
from simulator.baselines import volatility_targeting, no_hedge_policy, momentum_policy, delta_hedge_policy

# Build environment on the full panel (policy/reporting can subset by date if you prefer)
env = HedgingEnv(
    df=panel,
    features=state_cols,
    reward_fn=pnl_only,     # PnL net of costs (env charges txn_cost_bps)
    window=60,
    txn_cost_bps=1.0,       # baseline cost; adjust as needed or wire a cost_fn in your script
    scaler=scaler,
    hold_on_nan=True,       # IMPORTANT: don't trade when features are incomplete
)

# Simple baseline: vol targeting (uses vix if available)
feat_idx = state_cols.index("vix") if "vix" in state_cols else 0
policy = volatility_targeting(feature_idx=feat_idx, ann_vol_target=0.15)

res = env.rollout(policy)

rewards = np.asarray(res["rewards"], float)
sr = (rewards.mean() / rewards.std(ddof=1) * np.sqrt(252)) if rewards.std(ddof=1) > 0 else 0.0
print(f"Overlay Sharpe (full sample): {sr:.3f}")


Overlay Sharpe (full sample): 0.279


In [26]:
# --- Align sample to the first date with complete state features ---
def first_full_date(df, cols):
    m = df[cols].notna().all(axis=1)
    return df.loc[m, "date"].min()

cut = first_full_date(panel, state_cols)
panel_cut = panel.loc[panel["date"] >= cut].reset_index(drop=True)

print("Cut start:", cut, "| old rows:", len(panel), "-> new rows:", len(panel_cut))

# Rebuild env on the trimmed panel (scaler still fit on TRAIN window within this new panel)
TRAIN_END = pd.Timestamp("2017-12-31")
mu = panel_cut.loc[panel_cut["date"] <= TRAIN_END, state_cols].mean()
sigma = panel_cut.loc[panel_cut["date"] <= TRAIN_END, state_cols].std(ddof=1).replace(0, np.nan).fillna(1.0)
scaler = lambda obs: (obs - mu.values) / sigma.values

from simulator.env import HedgingEnv
env = HedgingEnv(
    df=panel_cut, features=state_cols, reward_fn=pnl_only,
    window=60, txn_cost_bps=1.0, scaler=scaler, hold_on_nan=True
)

res = env.rollout(policy)
r = np.asarray(res["rewards"], float)
print("Sharpe (trimmed):", (r.mean()/r.std(ddof=1)*np.sqrt(252)) if r.std(ddof=1)>0 else 0.0)


Cut start: 2005-03-09 00:00:00 | old rows: 18018 -> new rows: 17986
Sharpe (trimmed): 0.2789871270096591


In [27]:
# --- 4) (Optional) Evaluate by period (train / valid / test) ---

def period_sharpe(panel, rewards, start, end):
    # align rewards to panel rows the env stepped through
    n = len(rewards)
    dates = panel["date"].iloc[:n]
    mask = (dates >= start) & (dates <= end)
    r = rewards[mask.to_numpy()]
    if r.size < 5 or np.isclose(r.std(ddof=1), 0.0):
        return 0.0
    return r.mean() / r.std(ddof=1) * np.sqrt(252)

print("Sharpe train:", f"{period_sharpe(panel, rewards, panel['date'].min(), TRAIN_END):.3f}")
print("Sharpe valid:", f"{period_sharpe(panel, rewards, TRAIN_END + pd.Timedelta(days=1), VALID_END):.3f}")
print("Sharpe test :", f"{period_sharpe(panel, rewards, VALID_END + pd.Timedelta(days=1), panel['date'].iloc[len(rewards)-1]):.3f}")


Sharpe train: 0.269
Sharpe valid: 0.138
Sharpe test : 0.353


In [28]:
# --- 5) Quick sanity checks (catch silent drift early) ---

# Dates monotonic and forward returns present
assert panel["date"].is_monotonic_increasing
assert panel["ret_fwd"].notna().all()

# Feature completeness snapshot
na_counts = panel[state_cols].isna().sum().sort_values(ascending=False)
print("Top missing in state cols:\n", na_counts.head(8))

# First observation (after warmup) is finite when trading starts
obs0 = env.reset()
assert np.isfinite(obs0).all(), "NaN/Inf in initial observation; check staleness guard & scaler."


Top missing in state cols:
 iv_ts_slope_spx    625
iv_atm_30d_spx     601
iv_skew_30d_spx    601
vix                  0
rate_10y             0
rv_21d               0
hvol_30d             0
hvol_91d             0
dtype: int64
